<a href="https://colab.research.google.com/github/LuanBui1801/BT_MALNUTRITION/blob/main/Malnutrition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
import sys
import time

print("⏳ [1/4] Đang dọn dẹp bộ nhớ và tiêu diệt luồng mạng cũ...")
os.system("pkill -9 ngrok")
os.system("pkill -f streamlit")
time.sleep(2) # Đợi hệ thống giải phóng cổng 8501

print("⏳ [2/4] Đang cài đặt thư viện cần thiết...")
os.system("pip install -q streamlit pyngrok seaborn matplotlib scikit-learn pandas numpy")
print("✅ Cài đặt xong!")

print("⏳ [3/4] Đang tạo file mã nguồn giao diện (app.py)...")
app_code = """import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Perceptron
from sklearn.preprocessing import StandardScaler
import os

st.set_page_config(page_title="AI Diagnosis Pro Max", layout="wide")

# CSS: Giao diện Kính mờ (Glassmorphism) & Tông Đen Neon
st.markdown('''
<style>
    .stApp { background: linear-gradient(135deg, #050508, #120e24, #08121a); color: #f1f5f9; }
    h1 { background: linear-gradient(45deg, #00f2fe, #4facfe, #00ff87); -webkit-background-clip: text; -webkit-text-fill-color: transparent; text-align: center; font-weight: 900; text-shadow: 0px 0px 20px rgba(0,242,254,0.3); }
    .glass-box { background: rgba(255, 255, 255, 0.05); padding: 25px; border-radius: 16px; border: 1px solid rgba(255,255,255,0.1); }
</style>
''', unsafe_allow_html=True)

st.title("🤖 AI MALNUTRITION DIAGNOSTIC")
st.markdown("<p style='text-align:center; color:#94a3b8;'>Hệ thống dự đoán bằng mạng Nơ-ron Perceptron thuần túy</p><hr>", unsafe_allow_html=True)

# Kiểm tra file dữ liệu
if not os.path.exists("/content/malnutrition_data.csv"):
    st.error("❌ Không tìm thấy file 'malnutrition_data.csv'. Hãy kiểm tra lại tab thư mục!")
    st.stop()

@st.cache_data
def load_data():
    return pd.read_csv("/content/malnutrition_data.csv")

df = load_data()

# 5 Inputs: 'Age_months', 'Weight_kg', 'Height_cm', 'Sleep_hours', 'Meals_per_day'
X = df[['Age_months', 'Weight_kg', 'Height_cm', 'Sleep_hours', 'Meals_per_day']]
y = df['Diagnosis'] # Chứa >= 3 nhãn đầu ra
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Huấn luyện mô hình Perceptron đa lớp (One-Vs-Rest chạy ngầm trong sklearn)
@st.cache_resource
def get_model():
    model = Perceptron(max_iter=1000, random_state=42)
    model.fit(X_scaled, y)
    return model

model = get_model()

# --- SIDEBAR: 5 INPUTS ---
st.sidebar.header("⚙️ CHỈ SỐ ĐẦU VÀO")
age = st.sidebar.slider("Tuổi (tháng)", int(X['Age_months'].min()), int(X['Age_months'].max()), 24)
weight = st.sidebar.slider("Cân nặng (kg)", float(X['Weight_kg'].min()), float(X['Weight_kg'].max()), 12.0, 0.1)
height = st.sidebar.slider("Chiều cao (cm)", float(X['Height_cm'].min()), float(X['Height_cm'].max()), 85.0, 0.1)
sleep = st.sidebar.slider("Giờ ngủ/ngày", float(X['Sleep_hours'].min()), float(X['Sleep_hours'].max()), 10.0, 0.1)
meals = st.sidebar.slider("Bữa ăn/ngày", int(X['Meals_per_day'].min()), int(X['Meals_per_day'].max()), 4)

# Tính toán dự đoán
input_data = scaler.transform([[age, weight, height, sleep, meals]])
pred = model.predict(input_data)[0]

# Điểm quyết định định vị hình học tuyến tính
scores = model.decision_function(input_data)[0]

# SỬA LỖI: Chuẩn hóa điểm số hiển thị an toàn không lo lỗi mảng nhiều phần tử
if len(model.classes_) > 2:
    min_s, max_s = np.min(scores), np.max(scores)
    norm_scores = (scores - min_s) / (max_s - min_s + 1e-5) if max_s != min_s else np.ones_like(scores) * 0.5
else:
    # Trường hợp nhị phân (chỉ có 1 giá trị score đơn lẻ)
    norm_scores = [1.0 if scores >= 0 else 0.0]

# --- HIỂN THỊ KẾT QUẢ & BIỂU ĐỒ ---
col1, col2 = st.columns(2, gap="large")

with col1:
    st.markdown("### 🎯 KẾT QUẢ PHÂN LOẠI")

    # Đổi màu động theo nhãn kết quả dự đoán
    color = "#00ff87" if "Normal" in pred else "#ffbb00" if "Mild" in pred else "#ff3366"

    st.markdown(f'''
    <div style="background: rgba(255,255,255,0.03); border: 2px solid {color}; border-radius: 15px; padding: 30px; text-align: center; box-shadow: 0px 0px 25px {color}40;">
        <h2 style="color: {color}; margin: 0; font-size: 2.5rem;">{pred.upper()}</h2>
    </div>
    ''', unsafe_allow_html=True)

    st.markdown("<br><b>Mức độ tin cậy tuyến tính (Decision Score):</b><br><small style='color:#94a3b8;'>Perceptron phân tách bằng siêu phẳng. Điểm số càng lớn thể hiện khoảng cách đến ranh giới phân tách càng xa.</small>", unsafe_allow_html=True)

    for cls, score, norm_s in zip(model.classes_, scores, norm_scores):
        st.write(f"- **{cls}**: `Score = {score:.2f}`")
        st.progress(float(np.clip(norm_s, 0.0, 1.0))) # Đảm bảo giá trị luôn nằm trong khoảng [0.0, 1.0]

with col2:
    st.markdown("### 📈 BIỂU ĐỒ TƯƠNG QUAN THỂ TRẠNG")
    fig, ax = plt.subplots(figsize=(6, 4))
    fig.patch.set_facecolor('#050508')
    ax.set_facecolor('#111')

    # Vẽ phân tán dữ liệu nền
    sns.scatterplot(data=df, x='Height_cm', y='Weight_kg', hue='Diagnosis', ax=ax, alpha=0.6)

    # Vẽ điểm nhấn cho bệnh nhân đang kéo thanh Slider
    ax.scatter(height, weight, color='#00f2fe', s=350, marker='*', edgecolor='white', linewidth=1.5, label='Bệnh nhân này', zorder=10)

    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('#888')
    ax.yaxis.label.set_color('#888')
    ax.legend(facecolor='#111', labelcolor='white')
    st.pyplot(fig)

st.markdown("---")
st.markdown("### 🌡️ HEATMAP MA TRẬN HỆ SỐ TƯƠNG QUAN (5 INPUTS)")
fig2, ax2 = plt.subplots(figsize=(10, 3.5))
fig2.patch.set_facecolor('#050508')
ax2.set_facecolor('#111')
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='mako', ax=ax2, annot_kws={'color':'white', 'weight':'bold'})
ax2.tick_params(colors='white')
st.pyplot(fig2)
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)
print("✅ Tạo file thành công!")

print("⏳ [4/4] Khởi động luồng Ngrok và Streamlit...")
from pyngrok import ngrok

# Cấu hình Token Ngrok
ngrok.set_auth_token("3DimuhoA0CTMeThz8tcgyciZHXX_2pr6N3B2ZPF4jd5HAnmkx")
ngrok.kill()
time.sleep(2)

# Khởi chạy ngầm Streamlit
os.system("streamlit run app.py --server.port 8501 --server.headless true &")
time.sleep(5)

# Kết nối lấy URL public công cộng
try:
    public_url = ngrok.connect(8501)
    print("\n" + "="*70)
    print("🎉 WEBSITE PRO MAX ĐÃ CHẠY THÀNH CÔNG! HÃY CLICK VÀO LINK BÊN DƯỚI:")
    print(f"👉 {public_url.public_url}")
    print("="*70)
except Exception as e:
    print(f"❌ Có lỗi kết nối Ngrok: {e}")

⏳ [1/4] Đang dọn dẹp bộ nhớ và tiêu diệt luồng mạng cũ...
⏳ [2/4] Đang cài đặt thư viện cần thiết...
✅ Cài đặt xong!
⏳ [3/4] Đang tạo file mã nguồn giao diện (app.py)...
✅ Tạo file thành công!
⏳ [4/4] Khởi động luồng Ngrok và Streamlit...

🎉 WEBSITE PRO MAX ĐÃ CHẠY THÀNH CÔNG! HÃY CLICK VÀO LINK BÊN DƯỚI:
👉 https://slander-plow-trustee.ngrok-free.dev
